# Week 11 — Kalibrasi dan anotasi manusia

Jalankan notebook preprocessing dahulu. Notebook ini menyiapkan lembar kosong,
menampilkan progres pengisian, serta menyediakan langkah audit dan finalisasi
yang **tidak aktif secara default**. Tidak ada label dibuat oleh model.

Keempat anggota mengisi 50 kalibrasi secara mandiri, membahas perbedaan,
memperbaiki panduan, kemudian mengisi gold. Jangan menilai kesepakatan dari
label yang disalin bersama. Pilih kernel environment proyek.

## 1. Lingkungan dan sumber

In [ ]:
from pathlib import Path
import csv, hashlib, io, json, sys
from collections import Counter
from html import escape
from IPython.display import display, HTML

# Berjalan dari akar repo maupun folder notebooks, tanpa path C: tertentu.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/preprocess.py").is_file() and (p / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Buka notebook dari checkout repository Web Mining.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src import preprocess, annotation

def read_csv(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

def show_table(rows, columns=None, limit=10):
    rows = list(rows)
    if not rows:
        print("Tidak ada baris.")
        return
    columns = columns or list(rows[0])
    header = "".join(f"<th>{escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(r.get(c, '')))}</td>"
                   for c in columns) + "</tr>" for r in rows[:limit])
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

print("Root:", ROOT)
print("Python:", sys.version.split()[0])

In [ ]:
CLEAN_PATH = ROOT / "data/processed/reviews_clean.csv"
SPLIT_PATH = ROOT / "data/annotations/split.csv"
WORKSPACE = ROOT / "data/annotation_workspace"
if not CLEAN_PATH.exists() or not SPLIT_PATH.exists():
    raise FileNotFoundError("Jalankan 01_preprocessing.ipynb terlebih dahulu.")
baseline = json.loads((ROOT / "results/preprocessing/v1/summary.json").read_text())
assert annotation.digest(CLEAN_PATH) == baseline["outputs_sha256"]["reviews_clean"]
assert annotation.digest(SPLIT_PATH) == baseline["outputs_sha256"]["split"]
print("Sumber clean dan split sesuai versi preprocessing.")

## 2. Siapkan lembar tanpa menimpa anotasi

`prepare` membuat 50 kalibrasi dan 275 tugas gold untuk masing-masing anggota.
Jika workspace sudah ada, manifest diperiksa dan lembar yang mungkin sudah
diisi tidak disentuh. Workspace lama yang tidak lengkap perlu diperiksa,
bukan dibuat ulang diam-diam.

In [ ]:
if not WORKSPACE.exists():
    manifest = annotation.prepare(CLEAN_PATH, SPLIT_PATH, WORKSPACE)
    print("Lembar kosong dibuat.")
else:
    manifest_path = WORKSPACE / "manifest.json"
    if not manifest_path.is_file():
        raise ValueError("Workspace sudah ada tetapi manifest hilang; periksa manual.")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected = {"reviews_clean": annotation.digest(CLEAN_PATH), "split": annotation.digest(SPLIT_PATH)}
    if manifest["source_sha256"] != expected:
        raise ValueError("Workspace memakai sumber berbeda; jangan menimpa anotasi.")
    print("Workspace lama dipakai; lembar tidak diubah.")
show_table([{"Anggota": m, "Kalibrasi": len(manifest["assignments"]["kalibrasi"][m]),
             "Gold": len(manifest["assignments"]["gold"][m])} for m in annotation.MEMBERS])

## 3. Panduan dan progres pengisian

Enam aspek: akun_login, registrasi_data, fitur_kesehatan, performa,
kemudahan_penggunaan, umum. Pilihan label: tidak_disebut, positif, negatif,
netral. Umum hanya bila tidak ada aspek khusus. Jangan menggunakan rating
sebagai pengganti sentimen. Baca `data/annotations/README.md`.

Sel berikut hanya menghitung progres; label anggota lain tidak ditampilkan.

In [ ]:
progress = []
for phase in ["kalibrasi", "gold"]:
    for member in annotation.MEMBERS:
        path = WORKSPACE / f"{phase}_{member}.csv"
        rows = read_csv(path)
        filled = sum(all(r[a] in annotation.LABELS for a in annotation.ASPECTS) for r in rows)
        progress.append({"Fase": phase, "Anggota": member, "Baris lengkap": filled, "Total": len(rows)})
show_table(progress)
print("Isi lembar CSV masing-masing; jangan ubah ID, anotator, atau teks.")
print("Baris lengkap belum menjamin label valid secara semantik; audit tetap diperlukan.")

## 4. Audit setelah seluruh anotasi fase lengkap

Ubah RUN_AUDIT ke True hanya setelah empat lembar selesai. Pilih fase dan
folder audit baru. Label kosong akan ditolak. Audit menghasilkan kesepakatan
sebelum adjudikasi serta daftar konflik; ini bukan evaluasi model ABSA.

In [ ]:
RUN_AUDIT = False
PHASE = "kalibrasi"  # ganti ke "gold" setelah anotasi gold selesai
AUDIT_DIR = WORKSPACE / f"audit_{PHASE}_notebook_v1"
if RUN_AUDIT:
    audit_report = annotation.audit(WORKSPACE, PHASE, AUDIT_DIR)
    print("Ulasan:", audit_report["review_count"], "Sel konflik:", audit_report["conflict_cells"])
    show_table(audit_report["agreement"], limit=36)
else:
    print("Audit belum dijalankan. Tidak ada nilai kappa yang dihasilkan.")

## 5. Finalisasi setelah diskusi dan adjudikasi

Pertahankan label mandiri awal. Isi final_label, adjudicator, dan alasan pada
resolutions.csv. Konflik set uji harus diputuskan anggota ketiga.
RUN_FINALIZE tetap False sampai keputusan lengkap. Jangan memakai label
buatan AI sebagai label manusia. Hasil lama tidak ditimpa.

In [ ]:
RUN_FINALIZE = False
FINAL_DIR = ROOT / "data/annotations" / f"{PHASE}_notebook_v1"
RESOLUTIONS = AUDIT_DIR / "resolutions.csv"
if RUN_FINALIZE:
    final_report = annotation.finalize(WORKSPACE, PHASE, RESOLUTIONS, FINAL_DIR)
    print("Ekspor:", final_report["phase"], "Ulasan:", final_report["review_count"])
else:
    print("Finalisasi belum dijalankan. Gold final belum dibuat oleh notebook.")

## 6. Status pekerjaan

Lembar siap bukan berarti kalibrasi atau gold selesai. Setelah label manusia
final, Jap dapat melanjutkan pemodelan dan Geraldyn mengevaluasi prediksi pada
gold yang sama. Definisi metrik dan pemilihan konfigurasi mengikuti keputusan
metode. Workspace berisi teks/catatan tetap lokal dan diabaikan Git.